# 02 — Autograd

PyTorch tracks operations on tensors with `requires_grad=True` to build a computation graph, then computes gradients via `backward()`.

## Basic Gradient Computation

In [ ]:
import torch

x = torch.tensor(2.0, requires_grad=True)
y = x ** 2 + 3 * x + 1
y.backward()
x.grad          # dy/dx = 2x + 3 = 7


## Gradients over Vectors

In [ ]:
x = torch.randn(3, requires_grad=True)
y = (x ** 2).sum()
y.backward()
x.grad          # dy/dx = 2x


## Gradients Accumulate

Calling `backward()` multiple times adds to `.grad` — must zero it between steps (this is what `optimizer.zero_grad()` does).

In [ ]:
x = torch.tensor(1.0, requires_grad=True)
y = x * 2
y.backward()
print(x.grad)   # 2

y = x * 2
y.backward()
print(x.grad)   # 4 (accumulated!)

x.grad.zero_()  # reset


## Stopping Gradient Tracking

In [ ]:
x = torch.randn(3, requires_grad=True)

with torch.no_grad():
    y = x * 2          # not tracked, no grad_fn

y = x.detach() * 2     # detach from current graph, shares storage

for p in [x]:
    p.requires_grad_(False)   # in-place toggle


## Non-scalar backward (Jacobian-vector product)

`backward()` needs a scalar unless you pass a `gradient` argument matching the output shape (this represents upstream gradients, e.g. dL/dy).

In [ ]:
x = torch.randn(3, requires_grad=True)
y = x * 2
y.backward(torch.ones_like(y))   # equivalent to y.sum().backward()
x.grad


## Computation Graph Inspection

In [ ]:
x = torch.tensor(2.0, requires_grad=True)
y = x ** 2
y.grad_fn                 # <PowBackward0>
y.grad_fn.next_functions   # links back toward x
x.is_leaf                  # True — created directly, not from an op
y.is_leaf                  # False


## Custom autograd.Function

Define custom forward/backward when you need an operation autograd doesn't natively support.

In [ ]:
class Square(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x):
        ctx.save_for_backward(x)
        return x ** 2

    @staticmethod
    def backward(ctx, grad_output):
        x, = ctx.saved_tensors
        return grad_output * 2 * x

x = torch.tensor(3.0, requires_grad=True)
y = Square.apply(x)
y.backward()
x.grad   # 6.0


## Gradient Clipping (used during training to prevent exploding gradients)

In [ ]:
model = torch.nn.Linear(10, 1)
loss = model(torch.randn(4, 10)).sum()
loss.backward()
torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
torch.nn.utils.clip_grad_value_(model.parameters(), clip_value=0.5)


## Common Gotchas
- Forgetting `optimizer.zero_grad()` silently accumulates gradients across batches.
- Calling `.backward()` twice on the same graph without `retain_graph=True` raises an error (the graph is freed after use).
- In-place ops on tensors that require grad and are needed for the backward pass raise `RuntimeError`.
- `loss.item()` for logging — never keep the full tensor around across iterations (leaks the graph, blows up memory).

In [ ]:
x = torch.tensor(2.0, requires_grad=True)
y = x ** 2
y.backward(retain_graph=True)  # needed if you'll backward() through the same graph again
y.backward()
x.grad
